In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from pathlib import Path
import requests
from io import BytesIO
from zipfile import ZipFile, BadZipFile

import numpy as np
import pandas as pd
import pandas_datareader.data as web
from sklearn.datasets import fetch_openml

import requests
from io import StringIO

# pip install yfinance (설치되어 있지 않은 경우)
import yfinance as yf

pd.set_option('display.expand_frame_repr', False)

In [3]:
DATA_STORE = Path('data/assets.h5')

In [4]:
df = (pd.read_csv('data/WIKI_PRICES_989367082a7cf822fd35dbe3c8f4969b.csv',
                 parse_dates=['date'],
                 index_col=['date', 'ticker'],
                 infer_datetime_format=True)
     .sort_index())

print(df.info(show_counts=True))
with pd.HDFStore(DATA_STORE) as store:
    store.put('quandl/wiki/prices', df)

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 15389314 entries, (Timestamp('1962-01-02 00:00:00'), 'ARNC') to (Timestamp('2018-03-27 00:00:00'), 'ZUMZ')
Data columns (total 12 columns):
 #   Column       Non-Null Count     Dtype  
---  ------       --------------     -----  
 0   open         15388776 non-null  float64
 1   high         15389259 non-null  float64
 2   low          15389259 non-null  float64
 3   close        15389313 non-null  float64
 4   volume       15389314 non-null  float64
 5   ex-dividend  15389314 non-null  float64
 6   split_ratio  15389313 non-null  float64
 7   adj_open     15388776 non-null  float64
 8   adj_high     15389259 non-null  float64
 9   adj_low      15389259 non-null  float64
 10  adj_close    15389313 non-null  float64
 11  adj_volume   15389314 non-null  float64
dtypes: float64(12)
memory usage: 1.4+ GB
None


In [5]:
df = pd.read_csv('data/WIKI_PRICES_989367082a7cf822fd35dbe3c8f4969b.csv')
# no longer needed
# df = pd.concat([df.loc[:, 'code'].str.strip(),
#                 df.loc[:, 'name'].str.split('(', expand=True)[0].str.strip().to_frame('name')], axis=1)

print(df.info(show_counts=True))
with pd.HDFStore(DATA_STORE) as store:
    store.put('quandl/wiki/stocks', df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15389314 entries, 0 to 15389313
Data columns (total 14 columns):
 #   Column       Non-Null Count     Dtype  
---  ------       --------------     -----  
 0   ticker       15389314 non-null  object 
 1   date         15389314 non-null  object 
 2   open         15388776 non-null  float64
 3   high         15389259 non-null  float64
 4   low          15389259 non-null  float64
 5   close        15389313 non-null  float64
 6   volume       15389314 non-null  float64
 7   ex-dividend  15389314 non-null  float64
 8   split_ratio  15389313 non-null  float64
 9   adj_open     15388776 non-null  float64
 10  adj_high     15389259 non-null  float64
 11  adj_low      15389259 non-null  float64
 12  adj_close    15389313 non-null  float64
 13  adj_volume   15389314 non-null  float64
dtypes: float64(12), object(2)
memory usage: 1.6+ GB
None


In [6]:
df = web.DataReader(name='SP500', data_source='fred', start=2009).squeeze().to_frame('close')
print(df.info())
with pd.HDFStore(DATA_STORE) as store:
    store.put('sp500/fred', df)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2609 entries, 2016-09-06 to 2026-09-04
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   close   2514 non-null   float64
dtypes: float64(1)
memory usage: 40.8 KB
None


In [7]:
sp500_stooq = (pd.read_csv('data/^uslc_d.csv', index_col=0,
                     parse_dates=True).loc['1950':'2019'].rename(columns=str.lower))
print(sp500_stooq.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1710 entries, 2013-05-22 to 2019-12-31
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    1710 non-null   float64
 1   high    1710 non-null   float64
 2   low     1710 non-null   float64
 3   close   1710 non-null   float64
dtypes: float64(4)
memory usage: 66.8 KB
None


In [8]:
with pd.HDFStore(DATA_STORE) as store:
    store.put('sp500/stooq', sp500_stooq)

In [9]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

# 1. 마치 일반 크롬 브라우저로 접속하는 것처럼 User-Agent 정보를 설정합니다.
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

# 2. requests를 사용해 위키피디아 페이지의 HTML 내용을 가져옵니다.
response = requests.get(url, headers=headers)

# 3. 가져온 HTML 텍스트를 pandas의 read_html로 읽어옵니다.
# (StringIO를 사용하면 최신 Pandas 버전에서 발생하는 경고 메시지를 방지할 수 있습니다.)
df = pd.read_html(StringIO(response.text), header=0)[0]

# 데이터 확인
print(df.head())

  Symbol             Security             GICS Sector               GICS Sub-Industry    Headquarters Location  Date added      CIK      Founded
0    MMM                   3M             Industrials        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04    66740         1902
1    AOS          A. O. Smith             Industrials               Building Products     Milwaukee, Wisconsin  2017-07-26    91142         1916
2    ABT  Abbott Laboratories             Health Care           Health Care Equipment  North Chicago, Illinois  1957-03-04     1800         1888
3   ABBV               AbbVie             Health Care                   Biotechnology  North Chicago, Illinois  2012-12-31  1551152  2013 (1888)
4    ACN            Accenture  Information Technology  IT Consulting & Other Services          Dublin, Ireland  2011-07-06  1467373         1989


In [10]:
# 1. 현재 존재하는 8개 열에 맞춰서 이름 변경
df.columns = ['ticker', 'name', 'gics_sector', 'gics_sub_industry', 
              'location', 'first_added', 'cik', 'founded']

# 2. sec_filings 열은 애초에 없으니 drop을 생략하고 바로 ticker를 인덱스로 설정
df = df.set_index('ticker')

# 결과 확인
print(df.head())

                       name             gics_sector               gics_sub_industry                 location first_added      cik      founded
ticker                                                                                                                                        
MMM                      3M             Industrials        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04    66740         1902
AOS             A. O. Smith             Industrials               Building Products     Milwaukee, Wisconsin  2017-07-26    91142         1916
ABT     Abbott Laboratories             Health Care           Health Care Equipment  North Chicago, Illinois  1957-03-04     1800         1888
ABBV                 AbbVie             Health Care                   Biotechnology  North Chicago, Illinois  2012-12-31  1551152  2013 (1888)
ACN               Accenture  Information Technology  IT Consulting & Other Services          Dublin, Ireland  2011-07-06  1467373         1989

In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 503 entries, MMM to ZTS
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   name               503 non-null    object
 1   gics_sector        503 non-null    object
 2   gics_sub_industry  503 non-null    object
 3   location           503 non-null    object
 4   first_added        503 non-null    object
 5   cik                503 non-null    int64 
 6   founded            503 non-null    object
dtypes: int64(1), object(6)
memory usage: 31.4+ KB
None


In [12]:
with pd.HDFStore(DATA_STORE) as store:
    store.put('sp500/stocks', df)

In [13]:
# 1. 수동으로 다운로드한 CSV 파일 읽기 및 병합
files = ['data/nasdaq.csv', 'data/amex.csv', 'data/nyse.csv']
df = pd.concat([pd.read_csv(f) for f in files]).dropna(how='all', axis=1)

# 컬럼명 소문자로 변경 및 중복 제거
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '')

# 다운로드된 파일에 따라 심볼 컬럼명이 다를 수 있으므로 확인 후 인덱스 설정
if 'symbol' in df.columns:
    df = df.set_index('symbol')
elif 'ticker' in df.columns:
    df = df.set_index('ticker')

# 중복 인덱스 제거
df = df[~df.index.duplicated()]

# 2. 시가총액(Market Cap) 데이터를 숫자로 변환 (M, B 처리)
# 다운로드한 데이터의 marketcap이 문자열 형식이 아닌 경우를 대비해 문자열로 변환 후 처리
mcap = df[['marketcap']].dropna()
mcap['marketcap'] = mcap['marketcap'].astype(str) 

# 마지막 글자(M 또는 B) 추출
mcap['suffix'] = mcap.marketcap.str[-1]

# M(백만)과 B(십억)로 끝나는 데이터만 필터링
mcap = mcap[mcap.suffix.isin(['B', 'M'])].copy()

# 숫자 부분만 추출하여 float 형태로 변환 (예: "$8.7B" -> 8.7)
mcap['marketcap'] = mcap.marketcap.str.extract(r'([0-9.]+)').astype(float)

# M, B 단위에 맞춰 실제 숫자로 곱해주기
mcaps = {'M': 1e6, 'B': 1e9}
for symbol, factor in mcaps.items():
    mcap.loc[mcap.suffix == symbol, 'marketcap'] *= factor

# 변환된 시가총액 데이터를 원본 데이터프레임에 덮어쓰기
df['marketcap'] = mcap['marketcap']

# 3. HDF5 포맷으로 저장하기
DATA_STORE = 'data/assets.h5' # 저장할 HDF5 파일 이름 지정
with pd.HDFStore(DATA_STORE) as store:
    store.put('us_equities/stocks', df)

print("전처리 및 저장이 완료되었습니다!")
print(df.info())

전처리 및 저장이 완료되었습니다!
<class 'pandas.core.frame.DataFrame'>
Index: 7152 entries, AACG to ZWS
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       7152 non-null   object 
 1   lastsale   7152 non-null   object 
 2   netchange  7152 non-null   float64
 3   %change    7152 non-null   object 
 4   marketcap  0 non-null      float64
 5   country    6847 non-null   object 
 6   ipoyear    4129 non-null   float64
 7   volume     7152 non-null   int64  
 8   sector     6433 non-null   object 
 9   industry   6433 non-null   object 
dtypes: float64(3), int64(1), object(6)
memory usage: 614.6+ KB
None


In [14]:
mnist = fetch_openml('mnist_784', version=1)

In [15]:
print(mnist.DESCR)

**Author**: Yann LeCun, Corinna Cortes, Christopher J.C. Burges  
**Source**: [MNIST Website](http://yann.lecun.com/exdb/mnist/) - Date unknown  
**Please cite**:  

The MNIST database of handwritten digits with 784 features, raw data available at: http://yann.lecun.com/exdb/mnist/. It can be split in a training set of the first 60,000 examples, and a test set of 10,000 examples  

It is a subset of a larger set available from NIST. The digits have been size-normalized and centered in a fixed-size image. It is a good database for people who want to try learning techniques and pattern recognition methods on real-world data while spending minimal efforts on preprocessing and formatting. The original black and white (bilevel) images from NIST were size normalized to fit in a 20x20 pixel box while preserving their aspect ratio. The resulting images contain grey levels as a result of the anti-aliasing technique used by the normalization algorithm. the images were centered in a 28x28 image b

In [16]:
mnist.keys()

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])

In [17]:
mnist_path = Path('data/mnist')
if not mnist_path.exists():
    mnist_path.mkdir()

In [18]:
np.save(mnist_path / 'data', mnist.data.astype(np.uint8))
np.save(mnist_path / 'labels', mnist.target.astype(np.uint8))

In [19]:
fashion_mnist = fetch_openml(name='Fashion-MNIST')

In [20]:
print(fashion_mnist.DESCR)

**Author**: Han Xiao, Kashif Rasul, Roland Vollgraf  
**Source**: [Zalando Research](https://github.com/zalandoresearch/fashion-mnist)  
**Please cite**: Han Xiao and Kashif Rasul and Roland Vollgraf, Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms, arXiv, cs.LG/1708.07747  

Fashion-MNIST is a dataset of Zalando's article images, consisting of a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes. Fashion-MNIST is intended to serve as a direct drop-in replacement for the original MNIST dataset for benchmarking machine learning algorithms. It shares the same image size and structure of training and testing splits. 

Raw data available at: https://github.com/zalandoresearch/fashion-mnist

### Target classes
Each training and test example is assigned to one of the following labels:
Label  Description  
0  T-shirt/top  
1  Trouser  
2  Pullover  
3  Dress  
4  

In [21]:
label_dict = {0: 'T-shirt/top',
              1: 'Trouser',
              2: 'Pullover',
              3: 'Dress',
              4: 'Coat',
              5: 'Sandal',
              6: 'Shirt',
              7: 'Sneaker',
              8: 'Bag',
              9: 'Ankle boot'}

In [22]:
fashion_path = Path('data/fashion_mnist')
if not fashion_path.exists():
    fashion_path.mkdir()

In [23]:
pd.Series(label_dict).to_csv(fashion_path / 'label_dict.csv', index=False, header=None)

In [24]:
np.save(fashion_path / 'data', fashion_mnist.data.astype(np.uint8))
np.save(fashion_path / 'labels', fashion_mnist.target.astype(np.uint8))

In [25]:
# 1. FRED에서 조회 가능한 티커만 남기기 (GOLD 제외)
fred_securities = {
    'BAMLCC0A0CMTRIV'   : 'US Corp Master TRI',
    'BAMLHYH0A0HYM2TRIV': 'US High Yield TRI',
    'BAMLEMCBPITRIV'    : 'Emerging Markets Corporate Plus TRI',
    'DGS10'             : '10-Year Treasury CMR',
}

# 2. FRED 데이터 불러오기 (BAML 지수들은 최근 3년치만 제공됨)
df_fred = web.DataReader(name=list(fred_securities.keys()), data_source='fred', start=2000)
df_fred = df_fred.rename(columns=fred_securities)

# 3. 야후 파이낸스에서 금 대체 데이터 불러오기 (금 선물: GC=F)
gold_df = yf.download('GC=F', start='2000-01-01')

# yfinance 최신 버전에 따라 MultiIndex가 반환되는 경우를 대비한 처리
if isinstance(gold_df.columns, pd.MultiIndex):
    gold_series = gold_df['Close'].iloc[:, 0].rename('Gold (London, USD)')
else:
    gold_series = gold_df['Close'].rename('Gold (London, USD)')

# 4. 데이터 병합 및 샘플링
df = df_fred.join(gold_series, how='outer')
df = df.dropna(how='all').resample('B').mean()

# 5. HDF5 저장
DATA_STORE = 'data/assets.h5'
with pd.HDFStore(DATA_STORE) as store:
    store.put('fred/assets', df)
    
print("데이터 다운로드 및 병합 완료!")

c:\Users\hanyang\anaconda3\lib\site-packages\yfinance\scrapers\history.py:472: DeprecationWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
[*********************100%***********************]  1 of 1 completed


데이터 다운로드 및 병합 완료!
